# 22 — Relaciones temporales de C2 con OpenKOS: resultado de desarrollo

**Autor:** Jason Sepúlveda S.
**Proyecto:** Evaluación experimental de la extracción automática y la representación
persistente de decisiones organizacionales frente a Recuperación Aumentada (RAG)
**Depende de:** [`21-jss-extraccion-de-conceptos-openkos.ipynb`](21-jss-extraccion-de-conceptos-openkos.ipynb) ·
[ADR 0005](../docs/decisions/0005-adr-development-evaluation-split.md) ·
[ADR 0011](../docs/decisions/0011-adr-alineacion-rol-e-intervalos-del-experimento-a.md) ·
[ADR 0012](../docs/decisions/0012-adr-umbral-tau-del-experimento-a.md) ·
[ADR 0016](../docs/decisions/0016-adr-relaciones-temporales-de-openkos.md)

> **Estado: descriptivo, de desarrollo, sin resultados de la tesis.** Este notebook no llama a
> ningún modelo y no lee ninguna referencia: solo relee los dos reportes guardados
> `reports/experiment-a/{es2015,ts3009}-openkos070/relaciones.json`, que escribe
> `afg experiment a-relations`. Cubre dos series de desarrollo: **ES2015 con tres corridas** y
> **TS3009 con una**. **IS1004 no se puntuó**: la alineación de decisiones del Experimento A
> exige hoy una referencia adjudicada y su traducción al inglés, e IS1004 tiene un solo
> anotador y no está traducida (issue #204). Con **6 relaciones de oro** en las dos series
> (3 por serie), lo que sigue es solo una señal: sirve para ver que la medición funciona y
> dónde está el cuello de botella, no para comparar variantes (ADR 0016).

## Las métricas, en simple

| Término | Qué significa |
|---|---|
| Enlace | Cualquier relación entre dos decisiones que no sea «no relacionada» ni «introduce» (la decisión posterior plantea un objeto nuevo). |
| Relaciones de oro | Los enlaces de la Tarea B adjudicada por personas. Son la referencia. |
| Enlaces de la máquina | Los veredictos de `openkos revisions` que no son «sin relación». OpenKOS los juzga sobre las decisiones que extrajo la propia máquina. |
| Decisión alineada | Decisión de la máquina emparejada con una de la referencia por la regla del Experimento A (τ = 0,70). Un enlace solo puede acertar si sus dos decisiones están alineadas. |
| Principal, de extremo a extremo | Cuenta todas las relaciones de oro. Si sus decisiones no se extrajeron, la relación cuenta como omitida. |
| Condicionada a la alineación | Cuenta solo las relaciones de oro con ambas decisiones alineadas. Separa la calidad de las relaciones de la calidad de las decisiones (H2). |
| Tipo principal / estricto | Principal: «revierte» y «reemplaza» son una sola clase (la de OpenKOS, `reverses`). Estricto: `reverses` equivale a «revierte» solamente. |
| Dirección | Cuál de las dos decisiones es la posterior. Se deriva de las fechas de las reuniones, nunca del modelo. Principal: una dirección desconocida cuenta como error. Sensibilidad: esos pares salen del denominador. |
| TP, FP, FN | Aciertos, enlaces de la máquina que no corresponden a un enlace de oro y relaciones de oro no encontradas. |
| Intervalo de Wilson | Rango plausible de una proporción; con pocos casos es muy ancho. |

---

# 1. Datos

**Qué se hace y por qué.** Se cargan los dos reportes con una guarda que rechaza todo lo que no
sea de la partición de desarrollo (ADR 0005). Las tablas salen de `afg.experiments.relations_summary`.

In [1]:
import pandas as pd
from IPython.display import Markdown, display

from afg.experiments.relations_summary import (
    accuracy_frame,
    existence_frame,
    load_relation_report,
    run_frame,
)
from afg.shared.paths import EXPERIMENT_A_DIR

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)
pd.set_option("display.max_colwidth", 60)

RUN_NAMES = {"ES2015": "es2015-openkos070", "TS3009": "ts3009-openkos070"}
paths = {s: EXPERIMENT_A_DIR / name / "relaciones.json" for s, name in RUN_NAMES.items()}
missing = [str(p) for p in paths.values() if not p.exists()]
if missing:
    raise SystemExit(f"No se encontraron los reportes: {missing}\nVer notebooks/README.md.")

reports = {series: load_relation_report(path) for series, path in paths.items()}
runs = {series: run_frame(report) for series, report in reports.items()}
pooled = {series: report["pooled"] for series, report in reports.items()}

display(
    Markdown(
        "Series: "
        + ", ".join(
            f"**{s}** ({len(runs[s])} corrida{'s' if len(runs[s]) != 1 else ''}, "
            f"{reports[s]['n_errors']} con error, τ = {str(reports[s]['tau']).replace('.', ',')})"
            for s in reports
        )
        + "."
    )
)

Series: **ES2015** (3 corridas, 0 con error, τ = 0,7), **TS3009** (1 corrida, 0 con error, τ = 0,7).

---

# 2. Decisiones alineadas y enlaces

**Pregunta.** ¿Cuántas de las decisiones de la referencia encontró la máquina, y cuántos
enlaces detectó OpenKOS?

**Por qué importa.** Un enlace solo se puede puntuar si sus dos decisiones se alinean con la
referencia. Si pocas decisiones se alinean, casi ningún enlace de la máquina es evaluable.

In [2]:
for series, frame in runs.items():
    display(Markdown(f"### {series}"))
    display(frame.drop(columns="Serie").set_index("Corrida").T)

### ES2015

Corrida,run-0,run-1,run-2
Decisiones de la máquina,71,69,69
Decisiones de oro,27,27,27
Decisiones alineadas,9,9,8
Enlaces de oro,3,3,3
Enlaces de oro alineados,0,0,0
Enlaces de la máquina,26,26,23
reafirma,1,1,1
refina,17,18,17
revierte o reemplaza,8,7,5
sin relación,23,21,23


### TS3009

Corrida,run-0
Decisiones de la máquina,88
Decisiones de oro,26
Decisiones alineadas,13
Enlaces de oro,3
Enlaces de oro alineados,1
Enlaces de la máquina,26
reafirma,2
refina,17
revierte o reemplaza,7
sin relación,31


In [3]:
rows = []
for series, frame in runs.items():
    machine = frame["Enlaces de la máquina"].sum()
    aligned = frame["Enlaces con ambas decisiones alineadas"].sum()
    rows.append(
        {
            "Serie": series,
            "Corridas": len(frame),
            "Decisiones de oro": int(frame["Decisiones de oro"].iloc[0]),
            "Alineadas por corrida (mín. a máx.)": (
                f"{frame['Decisiones alineadas'].min()} a {frame['Decisiones alineadas'].max()}"
            ),
            "Enlaces de oro": int(frame["Enlaces de oro"].iloc[0]),
            "Enlaces de oro con ambas decisiones alineadas (máx. por corrida)": int(
                frame["Enlaces de oro alineados"].max()
            ),
            "Enlaces de la máquina por corrida (mín. a máx.)": (
                f"{frame['Enlaces de la máquina'].min()} a {frame['Enlaces de la máquina'].max()}"
            ),
            "Enlaces de la máquina con ambas decisiones alineadas": f"{aligned} de {machine}",
        }
    )
summary = pd.DataFrame(rows).set_index("Serie")
display(summary.T)

Serie,ES2015,TS3009
Corridas,3,1
Decisiones de oro,27,26
Alineadas por corrida (mín. a máx.),8 a 9,13 a 13
Enlaces de oro,3,3
Enlaces de oro con ambas decisiones alineadas (máx. por corrida),0,1
Enlaces de la máquina por corrida (mín. a máx.),23 a 26,26 a 26
Enlaces de la máquina con ambas decisiones alineadas,1 de 75,3 de 26


**Lectura.** OpenKOS detecta decenas de enlaces por corrida (la sección 6 da el rango), pero casi todos recaen sobre
decisiones de la máquina que **no se alinean** con la referencia. Esos enlaces no se pueden
puntuar como aciertos, así que cuentan como falsos positivos «sin alinear». El cuello de
botella no está en detectar relaciones sino en la **precisión de la extracción de decisiones**
(la máquina extrae muchas más decisiones que las de la referencia; ver la tabla).

---

# 3. Existencia del enlace

**Pregunta.** ¿Encuentra la máquina los enlaces de oro, y cuántos enlaces sobran?

**Método.** Principal: de extremo a extremo. Segunda cifra: condicionada a la alineación.
Cada tabla muestra cada corrida y el conjunto de las corridas de la serie (que suma los conteos:
en ES2015 las mismas 3 relaciones de oro se cuentan en cada una de las 3 corridas).

In [4]:
for series, report in reports.items():
    display(Markdown(f"### {series}"))
    frame = existence_frame(report)
    display(frame.set_index(["Ámbito", "Corrida"]))

### ES2015

TP  FP  FN                 Precisión                   Recall           F1  FP sin par candidato  FP con decisión sin alinear
Ámbito            Corrida                                                                                                                                
extremo a extremo run-0      0  26   3  0/26 (0,0 %) [0,0; 12,9]  0/3 (0,0 %) [0,0; 56,1]        0,000                     0                           26
                  run-1      0  26   3  0/26 (0,0 %) [0,0; 12,9]  0/3 (0,0 %) [0,0; 56,1]        0,000                     1                           25
                  run-2      0  23   3  0/23 (0,0 %) [0,0; 14,3]  0/3 (0,0 %) [0,0; 56,1]        0,000                     0                           23
                  conjunto   0  75   9   0/75 (0,0 %) [0,0; 4,9]  0/9 (0,0 %) [0,0; 29,9]        0,000                     1                           74
condicionada      run-0      0   0   0                 sin casos                sin casos  no definido                     0                            0
                  run-1      0   1   0   0/1 (0,0 %) [0,0; 79,3]                sin casos        0,000                     1                            0
                  run-2      0   0   0                 sin casos                sin casos  no definido                     0                            0
                  conjunto   0   1   0   0/1 (0,0 %) [0,0; 79,3]                sin casos        0,000                     1                            0

### TS3009

TP  FP  FN                 Precisión                       Recall     F1  FP sin par candidato  FP con decisión sin alinear
Ámbito            Corrida                                                                                                                              
extremo a extremo run-0      1  25   2  1/26 (3,8 %) [0,7; 18,9]     1/3 (33,3 %) [6,1; 79,2]  0,069                     2                           23
                  conjunto   1  25   2  1/26 (3,8 %) [0,7; 18,9]     1/3 (33,3 %) [6,1; 79,2]  0,069                     2                           23
condicionada      run-0      1   2   0  1/3 (33,3 %) [6,1; 79,2]  1/1 (100,0 %) [20,7; 100,0]  0,500                     2                            0
                  conjunto   1   2   0  1/3 (33,3 %) [6,1; 79,2]  1/1 (100,0 %) [20,7; 100,0]  0,500                     2                            0

Con el denominador condicionado, las corridas de ES2015 no tienen relaciones de oro con ambas
decisiones alineadas, así que el *recall* queda **sin casos** y el F1 **no definido** (salvo un cero en la corrida 1, donde
un enlace de la máquina cae en un par que la referencia no propuso): no hay cifra que reportar, y no se
completa con un cero.

---

# 4. Tipo y dirección

**Pregunta.** Cuando la máquina encuentra un enlace de oro, ¿acierta el tipo y cuál decisión es
la posterior?

**Método.** Cada exactitud se muestra sobre tres denominadores: todas las relaciones de oro
(extremo a extremo, principal), solo las que tienen ambas decisiones alineadas (condicionada)
y solo las que la máquina encontró (entre detectados). Las filas «estricta» y «desconocida
excluida» son las sensibilidades de ADR 0016.

In [5]:
for series, score in pooled.items():
    display(Markdown(f"### {series}"))
    display(accuracy_frame(score))

### ES2015

,Extremo a extremo,Condicionada,Entre detectados
Tipo (principal),"0/9 (0,0 %) [0,0; 29,9]",sin casos,sin casos
Tipo (estricta),"0/9 (0,0 %) [0,0; 29,9]",sin casos,sin casos
Dirección (principal),"0/9 (0,0 %) [0,0; 29,9]",sin casos,sin casos
Dirección (desconocida excluida),"0/9 (0,0 %) [0,0; 29,9]",sin casos,sin casos


### TS3009

,Extremo a extremo,Condicionada,Entre detectados
Tipo (principal),"1/3 (33,3 %) [6,1; 79,2]","1/1 (100,0 %) [20,7; 100,0]","1/1 (100,0 %) [20,7; 100,0]"
Tipo (estricta),"1/3 (33,3 %) [6,1; 79,2]","1/1 (100,0 %) [20,7; 100,0]","1/1 (100,0 %) [20,7; 100,0]"
Dirección (principal),"1/3 (33,3 %) [6,1; 79,2]","1/1 (100,0 %) [20,7; 100,0]","1/1 (100,0 %) [20,7; 100,0]"
Dirección (desconocida excluida),"1/3 (33,3 %) [6,1; 79,2]","1/1 (100,0 %) [20,7; 100,0]","1/1 (100,0 %) [20,7; 100,0]"


In [6]:
from afg.experiments.relations_summary import format_proportion

for series, score in pooled.items():
    display(
        Markdown(
            f"**{series}:** proporción de direcciones desconocidas entre los enlaces detectados: "
            f"{format_proportion(score['unknown_direction'])}."
        )
    )

**ES2015:** proporción de direcciones desconocidas entre los enlaces detectados: sin casos.

**TS3009:** proporción de direcciones desconocidas entre los enlaces detectados: 0/1 (0,0 %) [0,0; 79,3].

**Lectura.** En TS3009 el único enlace de oro alineado se encontró con el **tipo y la dirección
correctos**, tanto en el valor principal como en las sensibilidades (no hay direcciones
desconocidas entre los detectados). Es un solo caso: un intervalo de Wilson que va de poco más
del 20 % al 100 % impide decir algo más que «la medición funciona y no falló en el caso que
pudo evaluarse». En ES2015 no hay relaciones de oro con ambas decisiones alineadas, así que
tipo y dirección no tienen nada que medir fuera del denominador de extremo a extremo.

---

# 5. Falsos positivos sin par candidato (`not_candidate`)

**Pregunta.** Entre los enlaces de la máquina cuyas dos decisiones sí se alinean, ¿cuántos caen
en un par que la referencia no consideró candidato?

**Por qué importa.** Los pares candidatos de la referencia salen del filtro léxico. Un enlace
de la máquina sobre un par que el filtro no propuso se trata como «sin enlace» y cuenta como falso
positivo, pero puede ser una relación real que la referencia nunca vio (el *recall* del filtro
se mide aparte, en la Tarea C).

In [7]:
rows = []
for series, report in reports.items():
    for label, score in [(f"run-{e['run']}", e["score"]) for e in report["entries"]] + [
        ("conjunto", report["pooled"])
    ]:
        breakdown = score["existence_end_to_end"]["fp_breakdown"]
        rows.append(
            {
                "Serie": series,
                "Corrida": label,
                "FP sin alinear": breakdown["unaligned"],
                "FP en par sin candidato": breakdown["not_candidate"],
                "FP en par de oro sin enlace": breakdown["gold_no_link"],
            }
        )
display(pd.DataFrame(rows).set_index(["Serie", "Corrida"]))

FP sin alinear  FP en par sin candidato  FP en par de oro sin enlace
Serie  Corrida                                                                       
ES2015 run-0                 26                        0                            0
       run-1                 25                        1                            0
       run-2                 23                        0                            0
       conjunto              74                        1                            0
TS3009 run-0                 23                        2                            0
       conjunto              23                        2                            0

---

# 6. Qué se concluye

In [8]:
es, ts = pooled["ES2015"], pooled["TS3009"]
er, tr = runs["ES2015"], runs["TS3009"]
gold_links = er["Enlaces de oro"].iloc[0] + tr["Enlaces de oro"].iloc[0]
links = pd.concat([er["Enlaces de la máquina"], tr["Enlaces de la máquina"]])
ts_machine_dec = int(tr["Decisiones de la máquina"].iloc[0])
ts_gold_dec = int(tr["Decisiones de oro"].iloc[0])
ts_aligned_dec = int(tr["Decisiones alineadas"].iloc[0])
es_gold_dec = int(er["Decisiones de oro"].iloc[0])
es_min, es_max = er["Decisiones alineadas"].min(), er["Decisiones alineadas"].max()
es_tp, ts_tp = es["existence_end_to_end"]["tp"], ts["existence_end_to_end"]["tp"]
display(
    Markdown(f"""
- **Desarrollo solamente.** ES2015 con {len(er)} corridas y TS3009 con {len(tr)};
  IS1004 no se puntuó (issue #204). Con {gold_links} relaciones de oro entre las dos
  series, todo es una señal y no una estimación.
- **La detección de relaciones no es el cuello de botella.** OpenKOS propone entre
  {links.min()} y {links.max()} enlaces por corrida, pero solo
  {es["n_machine_links_aligned"]} de {es["n_machine_links"]} (ES2015, tres corridas) y
  {ts["n_machine_links_aligned"]} de {ts["n_machine_links"]} (TS3009) tienen sus dos
  decisiones alineadas con la referencia.
- **El cuello de botella es la precisión de la extracción de decisiones:** en TS3009 la
  máquina extrae {ts_machine_dec} decisiones donde la referencia tiene {ts_gold_dec}, y se
  alinean {ts_aligned_dec}; en ES2015 se alinean entre {es_min} y {es_max} de {es_gold_dec}.
- **Valor principal de extremo a extremo:** ES2015 {es_tp} aciertos de
  {es["n_gold_links"]} enlaces de oro en total; TS3009 {ts_tp} de {ts["n_gold_links"]}.
- **TS3009:** el único enlace de oro alineado se encontró con tipo y dirección correctos.
""")
)


- **Desarrollo solamente.** ES2015 con 3 corridas y TS3009 con 1;
  IS1004 no se puntuó (issue #204). Con 6 relaciones de oro entre las dos
  series, todo es una señal y no una estimación.
- **La detección de relaciones no es el cuello de botella.** OpenKOS propone entre
  23 y 26 enlaces por corrida, pero solo
  1 de 75 (ES2015, tres corridas) y
  3 de 26 (TS3009) tienen sus dos
  decisiones alineadas con la referencia.
- **El cuello de botella es la precisión de la extracción de decisiones:** en TS3009 la
  máquina extrae 88 decisiones donde la referencia tiene 26, y se
  alinean 13; en ES2015 se alinean entre 8 y 9 de 27.
- **Valor principal de extremo a extremo:** ES2015 0 aciertos de
  9 enlaces de oro en total; TS3009 1 de 3.
- **TS3009:** el único enlace de oro alineado se encontró con tipo y dirección correctos.
